# Lab L1.3 · Transaction feature factory — 1.5M card transactions into customer features (and images into numbers)
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 1 · ⏱ 50 min · timetabled Day 2, 15:40 · steps 1–5 & 7 (step 6 optional) · **Level 2 · Guided starter (fill in the blanks) · ★ recommended level**

Load 1.5M transactions memory-efficiently, aggregate them into a 250k-row customer feature table, pick the features that carry signal for income estimation, and turn unstructured document images into structured features.

### How this notebook works
- Every `✏️ BLANK n` marks code **you** write. Inline blanks look like `____`; block blanks say `# ...your code here...`.
- The markdown cell above each code cell explains **what** to write, **where**, and **why**.
- Stuck > 5 min? Open the 💡 hint in the HTML lab guide, then ask your trainer. The Level 3 solution is released after the lab.

### Scenario
**Horizon Gulf Bank (HGB)** is a fictional UAE retail bank with 250,000 customers. Its Board approved **Project MIZAN** — an AI decisioning platform covering credit, fraud, collateral valuation, cash logistics, customer retention and document automation. You are a junior ML practitioner in the MIZAN squad.

The Credit Risk team wants an **income estimation** model for customers whose salary is *not* paid into HGB (tomorrow's Lab L2.3). Card behaviour is the richest signal the bank holds about those customers, but it lives in 1.5 million transaction rows, not in one row per customer. Your job: build the **customer-level transaction feature table**, decide which features are worth keeping, and show the KYC team how unstructured images can be described by numbers.

**Progressive architecture:** joins onto your `customers_clean.parquet` (L1.1); the feature table feeds income estimation (L2.3) and churn modelling (L3.4, L4.3, L5.3).

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Load 1.5M rows the memory-smart way

**Why this matters.** Pandas stores every repeated string (customer id, merchant category, channel) as a separate Python object. On 1.5M rows that is hundreds of MB. Reading **only the columns you need**, storing repeated strings as `category` and amounts as `float32` usually cuts memory by 70–90% — the difference between a notebook that runs on a laptop and one that crashes. Saving to **Parquet** is columnar and compressed, and — unlike CSV — remembers the dtypes, so nobody has to re-specify them.

#### ✏️ What to write in the next cell
- **Blank 1** *(inline)* — only the columns the features need
- **Blank 2** *(inline)* — category stores each distinct string once

In [ ]:
t0 = time.time()
naive = pd.read_csv(DATA / "card_transactions.csv.gz")
t_naive, mb_naive = time.time() - t0, naive.memory_usage(deep=True).sum() / 1e6
del naive
DTYPES = {"customer_id": "category", "merchant_category": "category", "channel": "category",
          "merchant_country": "category", "amount_aed": "float32", "is_fraud": "int8"}
t0 = time.time()
tx = pd.read_csv(DATA / "card_transactions.csv.gz", usecols=____,  # ✏️ B1
                 dtype=____, parse_dates=["txn_ts"])  # ✏️ B2
t_smart, mb_smart = time.time() - t0, tx.memory_usage(deep=True).sum() / 1e6
print(f"naive: {mb_naive:,.0f} MB in {t_naive:.1f}s | smart: {mb_smart:,.0f} MB in {t_smart:.1f}s "
      f"→ {1 - mb_smart / mb_naive:.0%} less memory")
tx.to_parquet(CKPT / "card_transactions.parquet", index=False)
tx = pd.read_parquet(CKPT / "card_transactions.parquet")
size = lambda p: p.stat().st_size / 1e6
print(f"csv.gz {size(DATA / 'card_transactions.csv.gz'):.0f} MB on disk vs parquet {size(CKPT / 'card_transactions.parquet'):.0f} MB "
      f"| dtypes survive the round-trip: {sorted(set(tx.dtypes.astype(str)))}")
print(f"{len(tx):,} rows | {tx.customer_id.nunique():,} card customers")

**✔ Checkpoint.** Naive ≈ 320 MB; smart ≈ 33 MB (≈ 89% less). Parquet is ≈ 19 MB on disk versus 44 MB for the gzipped CSV, and the `category`/`float32` dtypes survive the round-trip. 1,506,430 rows from 58,822 card customers.

## Step 2 · Aggregate events into one row per customer

**Why this matters.** Models need one row per **entity** (customer), but behaviour arrives as **events** (transactions). Aggregation is the core feature-engineering move: *recency, frequency, monetary* (RFM) plus behaviour **shares** (night, international, e-commerce) and **diversity** (distinct categories/countries). Confirmed fraud is excluded — it is the fraudster's behaviour, not the customer's. Everything is computed **as of 1 Oct 2025** so no future information leaks in.

#### ✏️ What to write in the next cell
- **Blank 3** *(inline)* — named aggregation: new_column=(source_column, function)
- **Blank 4** *(inline)* — days since the last purchase, measured at the as-of date

In [ ]:
AS_OF = pd.Timestamp("2025-10-01")
tx = tx[tx.is_fraud == 0].copy()
tx["log_amt"] = np.log1p(tx.amount_aed)
tx["is_intl"] = (tx.merchant_country != "AE").astype("int8")
tx["is_night"] = tx.txn_ts.dt.hour.between(0, 5).astype("int8")
tx["is_ecom"] = (tx.channel == "E-Commerce").astype("int8")
g = tx.groupby("customer_id", observed=True)
agg = ____(  # ✏️ B3
    txn_count=("amount_aed", "size"), spend_total=("amount_aed", "sum"), amt_median=("amount_aed", "median"),
    log_amt_mean=("log_amt", "mean"), log_amt_std=("log_amt", "std"),
    intl_share=("is_intl", "mean"), night_share=("is_night", "mean"), ecom_share=("is_ecom", "mean"),
    n_mcc=("merchant_category", "nunique"), n_countries=("merchant_country", "nunique"),
    first_ts=("txn_ts", "min"), last_ts=("txn_ts", "max"))
agg["amt_p90"] = g.amount_aed.quantile(0.9)
agg["recency_days"] = ____  # ✏️ B4
active_months = ((agg.last_ts - agg.first_ts).dt.days / 30.4).clip(lower=1)
agg["txn_per_month"] = agg.txn_count / active_months
agg = agg.drop(columns=["first_ts", "last_ts"])
print(agg.shape); agg.describe().T.round(2)

**✔ Checkpoint.** 58,798 rows × 13 columns. Median 19 transactions per customer; `intl_share` ≈ 0.12; median `recency_days` 6. `log_amt_std` has ≈ 1,500 NaNs — customers with a single transaction have no spread.

## Step 3 · Spend mix and trend features

**Why this matters.** *Where* someone spends says a lot about them: a high share on luxury goods, travel and electronics versus groceries and fuel is a classic affluence signal. A **pivot table** turns the long category column into one share column per category. A **trend** feature (last 60 days versus the first 60 days) captures customers who are winding down — useful for churn.

#### ✏️ What to write in the next cell
- **Blank 5** *(inline)* — row-normalise: each customer's spend shares sum to 1

In [ ]:
mix = tx.pivot_table(index="customer_id", columns="merchant_category", values="amount_aed",
                     aggfunc="sum", fill_value=0, observed=True)
mix = ____.add_prefix("mix_")  # ✏️ B5
early = tx[tx.txn_ts < pd.Timestamp("2025-06-01")].groupby("customer_id", observed=True).amount_aed.sum()
recent = tx[tx.txn_ts >= AS_OF - pd.Timedelta(days=60)].groupby("customer_id", observed=True).amount_aed.sum()
trend = np.log((recent.reindex(agg.index, fill_value=0) + 1) / (early.reindex(agg.index, fill_value=0) + 1)).rename("spend_trend")
feat = agg.join(mix).join(trend)
feat.index = feat.index.astype(str)
print(feat.shape, "| mix rows sum to 1:", np.allclose(mix.sum(axis=1), 1))
cols = ["txn_count", "spend_total", "mix_Luxury_Goods", "mix_Grocery", "spend_trend"]
display(feat.sort_values("mix_Luxury_Goods", ascending=False).head(5)[cols].round(2))
feat[feat.txn_count >= 10].sort_values("mix_Luxury_Goods", ascending=False).head(5)[cols].round(2)

**✔ Checkpoint.** 58,798 × 28 features; mix rows sum to 1. The first table is all customers with **one** transaction (share = 100%) — shares computed from tiny counts are noise. With ≥ 10 transactions the top luxury shares are ≈ 50–70%.

## Step 4 · Join to the customer master — missing means what?

**Why this matters.** Only ~59k of 250k customers hold an active card. After a **left join** the other 190k have no transaction features. For *counts and totals* missing truly means **zero**; for *averages and shares* it means **unknown** — filling it with 0 would claim these customers spend AED 1 on average. We fill each group correctly and add a `has_card_activity` flag so a model can tell the two apart.

#### ✏️ What to write in the next cell
- **Blank 6** *(inline)* — keep every customer, even without a card
- **Blank 7** *(code block, ≈3 lines)* — counts/totals → 0; averages/shares → median of card holders; recency → 999 days

In [ ]:
cust = pd.read_parquet(CKPT / "customers_clean.parquet")
X = cust[["customer_id"]].merge(feat, left_on="customer_id", right_index=True, how=____, validate="one_to_one")  # ✏️ B6
assert len(X) == len(cust) and X.customer_id.is_unique
X["has_card_activity"] = X.txn_count.notna().astype(int)
ZERO = ["txn_count", "spend_total", "txn_per_month", "n_mcc", "n_countries", "spend_trend"] + [c for c in X if c.startswith("mix_")]
UNKNOWN = ["amt_median", "amt_p90", "log_amt_mean", "log_amt_std", "intl_share", "night_share", "ecom_share"]
# ✏️ BLANK 7 — write ≈3 line(s): counts/totals → 0; averages/shares → median of card holders; recency → 999 days
# ...your code here...
print(f"card activity: {X.has_card_activity.mean():.1%} of customers | NaNs left: {int(X.isna().sum().sum())} | shape {X.shape}")

**✔ Checkpoint.** ≈ 23.5% of customers have card activity; 0 NaNs; 250,000 rows.

## Step 5 · Feature selection with a random probe

**Why this matters.** More features ≠ better model. Three cheap filters remove most junk: **variance** (constant columns carry nothing), **correlation pruning** (near-duplicates add noise and instability) and **mutual information** (any — not only linear — dependence on the target). The trick professionals use is a **random probe**: a column of pure noise. Any feature that scores below the probe is indistinguishable from noise. The target here is the *verified* salary (salary paid into HGB).

#### ✏️ What to write in the next cell
- **Blank 8** *(inline)* — correlated with an earlier column
- **Blank 9** *(inline)* — captures non-linear dependence too

In [ ]:
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression
pop = X.merge(cust[["customer_id", "monthly_salary_aed", "salary_transfer", "salary_missing"]], on="customer_id")
pop = pop[(pop.has_card_activity == 1) & (pop.salary_transfer == 1) & (pop.salary_missing == 0)]
FEATS = [c for c in X.columns if c not in ("customer_id", "has_card_activity")]
F = pop[FEATS].assign(probe_noise=np.random.default_rng(RNG).normal(size=len(pop)))
y = np.log(pop.monthly_salary_aed)
vt = VarianceThreshold(1e-6).fit(F)
F = F.loc[:, vt.get_support()]
corr = F.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
drop_corr = [c for c in upper.columns if ____]  # ✏️ B8
F = F.drop(columns=drop_corr)
s = F.sample(20_000, random_state=RNG)
mi = pd.Series(____, index=F.columns).sort_values()  # ✏️ B9
KEEP = [c for c in mi.index if mi[c] > mi["probe_noise"]]
print(f"population {len(pop):,} | dropped (correlation): {drop_corr} | kept {len(KEEP)} of {len(mi) - 1}: {KEEP[::-1]}")
colors = ["tab:red" if c == "probe_noise" else ("tab:blue" if c in KEEP else "lightgrey") for c in mi.index]
mi.plot.barh(color=colors, figsize=(8, 8), title="Mutual information with log salary (red = random probe)"); plt.show()

**✔ Checkpoint.** ≈ 34.6k training customers. `txn_per_month` is pruned (near-duplicate of `txn_count`). 8 of 27 features beat the probe: `log_amt_mean`, `amt_median`, `amt_p90`, `spend_total` lead, followed by cash-withdrawal share, `n_mcc`, e-commerce share and money-transfer share. **Surprise:** luxury and travel shares score *at or below noise* — the ‘affluence’ intuition from step 3 is not supported by this data. Evidence beats intuition.

## Step 6 · Unstructured data → structured features

**Why this matters.** Images, text and audio are **unstructured**: there are no columns. The classical route is to *engineer* numeric descriptors — ink density, edge strength, where the ink sits on the page — and feed those to a normal model. On Day 4 a CNN will learn its own features from raw pixels; today's numbers are the baseline it has to beat.

#### ✏️ What to write in the next cell
- **Blank 10** *(inline)* — change between neighbouring columns = vertical strokes

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
docs = np.load(DATA / "kyc_documents_64.npz"); imgs, labels, classes = docs["X"] / 255.0, docs["y"], docs["classes"]
ink = 1 - imgs                                            # dark pixels = ink
rows = ink.mean(axis=2)
img_feat = pd.DataFrame({
    "ink_mean": ink.mean(axis=(1, 2)), "ink_std": ink.std(axis=(1, 2)),
    "edge_h": ____,  # ✏️ B10
    "edge_v": np.abs(np.diff(imgs, axis=1)).mean(axis=(1, 2)),
    "ink_top": ink[:, :32].mean(axis=(1, 2)), "ink_bottom": ink[:, 32:].mean(axis=(1, 2)),
    "ink_left": ink[:, :, :32].mean(axis=(1, 2)), "ink_right": ink[:, :, 32:].mean(axis=(1, 2)),
    "text_rows": (rows > rows.mean(axis=1, keepdims=True)).sum(axis=1)})
Xtr, Xte, ytr, yte = train_test_split(img_feat, labels, test_size=0.2, stratify=labels, random_state=RNG)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)
print(f"9 hand-made features → accuracy {clf.score(Xte, yte):.1%} (majority baseline 20%)")
img_feat.assign(doc=classes[labels]).groupby("doc").mean().round(3)

**✔ Checkpoint.** Hand-made features reach ≈ 67% accuracy — far above the 20% baseline but well below the ≈ 97% the Day 4 CNN achieves by learning its own features.

## Step 7 · Persist the feature table and its contract

**Why this matters.** Tomorrow's lab must use *exactly* these features with *exactly* these fill rules. We save the table plus a small JSON **feature spec** recording the as-of date, the selection evidence and the kept list — the seed of a feature store.

In [ ]:
import json
X.to_parquet(CKPT / "customer_txn_features.parquet", index=False)
spec = {"as_of": str(AS_OF.date()), "selected": KEEP, "dropped_correlated": drop_corr,
        "mutual_info": mi.round(4).to_dict(), "fill_rules": {"zero": ZERO, "card_holder_median": UNKNOWN, "recency_days": 999}}
(CKPT / "txn_feature_spec.json").write_text(json.dumps(spec, indent=2))
print(f"saved {X.shape} feature table; {len(KEEP)} selected features")

**✔ Checkpoint.** Feature table (250,000 × 30) and spec with 8 selected features saved.

## ✅ Lab complete
Transaction behaviour is now one tidy row per customer, with evidence for which features matter. **Next (Day 2):** tuning an income estimation model on it.